## Reconstruction of Waveforms from mel spectrograms

### Reconstruct with griffin lim default that produces warbling

In [35]:
import librosa 
import soundfile as sf
import numpy as np
from IPython.display import Audio
import json

try:
    with open('specmodel/config.json', 'r') as file:
        mel_config = json.load(file)
except FileNotFoundError:
    print("Error: Mel config.json was not found.")

stats = np.load('mel_stats.npz')
mel_mean = stats["mean"]   #(80,)
mel_std  = stats["std"]    #(80,)
mel_std = np.where(mel_std < 1e-6, 1e-6, mel_std)

#load preprocessed mel
songname = "0_Food_AWOL"
AUDIO_FILE_NPY = f'./data/mels/0-of-15/{songname}.npy'
mel = np.load(AUDIO_FILE_NPY)
mel = mel.T

print("Mel Shape", mel.shape)

Mel Shape (517, 80)


#### Padding from __getitem__()

In [36]:
TARGET_FRAMES = int(mel_config['clip_length'] * mel_config['sample_rate'] / mel_config['hop_length'])

if mel.shape[0] > TARGET_FRAMES:
    mel = mel[:TARGET_FRAMES]
elif mel.shape[0] < TARGET_FRAMES:
    pad_len = TARGET_FRAMES - mel.shape[0]
    pad = np.tile(mel_mean, (pad_len, 1))   # same as dataset
    mel = np.concatenate([mel, pad], axis=0)

#normalize
print(mel.shape)
mel = (mel - mel_mean) / mel_std

mel = mel.T #(freq x time)

print(mel.shape)
#unnorm
mel = mel * mel_std[:, None] + mel_mean[:, None]

mel = np.exp(mel)

(516, 80)
(80, 516)


In [37]:
audio = librosa.feature.inverse.mel_to_audio(
        mel,
        sr=mel_config['sample_rate'],
        n_fft=mel_config['n_fft'],
        hop_length=mel_config['hop_length'],
        n_iter=1000, #number of iterations for griffin lim
        win_length=mel_config['win_length'], 
        fmin=mel_config['fmin'], 
        #fmax=mel_config['fmax'], 
        power=mel_config['power']
    )

#ensure amplitudes are in safe range and doesn't blow speakers
audio /= (np.max(np.abs(audio)) + 1e-9)

sf.write(f"test_reconst_{songname}.wav", audio, mel_config['sample_rate'])

#### HIFI GAN reconstruction with Universal_V1 pretrained

In [60]:
import sys
sys.path.append("./hifi-gan")

import torch
import json
from models import Generator
from env import AttrDict

In [ ]:
# Load config
with open("hifi-gan/config_hifigan.json") as f:
    config = json.load(f)

h = AttrDict(config)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

generator = Generator(h).to(device)
checkpoint = torch.load("hifi-gan/gen_hifi_gan_02500000",
    map_location=device
)

generator.load_state_dict(checkpoint["generator"])
generator.eval()
generator.remove_weight_norm()

Removing weight norm...


In [62]:
print(sum(p.numel() for p in generator.parameters()) / 1e6, "M parameters")

13.926017 M parameters


In [70]:
samples, sample_rate = librosa.load(AUDIO_FILE_WF, sr=22050, mono=True) #consistent sampling rate and mono audio for all samples

mel_magnitude = librosa.feature.melspectrogram(
    y=samples,
    sr=sample_rate,
    n_fft=1024,
    hop_length=256,
    win_length=1024,
    fmin=0.0, 
    fmax=8000.0, 
    n_mels=80,
    power=1.0
)

mel_log = np.log(np.clip(mel_magnitude, 1e-5, None))
# Shape: (1, n_mels, T)
mel_tensor = torch.from_numpy(mel_log).unsqueeze(0).to(device)

print("Shape of mel_tensor: " , mel_tensor.size())

with torch.no_grad():
    audio_hifi = generator(mel_tensor).squeeze().cpu().numpy()

audio_hifi /= (np.max(np.abs(audio_hifi)) + 1e-9)

sf.write("reconstructed_hifigan.wav", audio_hifi, sample_rate)

Shape of mel_tensor:  torch.Size([1, 80, 2582])


Raw Audio Waveform

In [64]:
Audio(AUDIO_FILE_WF)

Reconstructed with Pretrained hifigan
80 mels

In [65]:
Audio('reconstructed_hifigan.wav')

Reconstructed with griff lim 1000 iters and 256 mels

In [66]:
Audio('reconstructed_grifflim_256_mels_1000_iters.wav')

#### Testing Griffin Lim with n_mels 80 and 1000 iterations

In [67]:
import librosa 
import soundfile as sf
import numpy as np
from IPython.display import Audio

AUDIO_FILE_WF = './data/waveforms/0-of-15/0_Food_AWOL.wav'
samples, sample_rate = librosa.load(AUDIO_FILE_WF, sr=22050, mono=True) #consistent sampling rate and mono audio for all samples

mel = librosa.feature.melspectrogram(
    y=samples,
    sr=sample_rate,
    n_fft=2048,
    hop_length=512,
    n_mels=80,
    power=2.0
)

mel_db = librosa.power_to_db(mel, top_db=80)

audio_test = librosa.feature.inverse.mel_to_audio(
    librosa.db_to_power(mel_db),
    sr=sample_rate,
    n_fft=2048,
    hop_length=512,
    n_iter=1000, 
    win_length=2048, 
    fmin=0.0, 
    fmax= sample_rate / 2
)
audio_test /= (np.max(np.abs(audio_test)) + 1e-9)

sf.write("reconstructed_grifflim_80_mels_1000_iters.wav", audio_test, 22050)

In [68]:
Audio('reconstructed_grifflim_80_mels_1000_iters.wav')